# PyCaret aplicado a problemas de classificação binária 

## Tutorial de utilização

* Demonstrar como a biblioteca PyCaret pode ser usada em problemas de classificação binária - credit scoring aplicado ao financiamento do veículo.

## Métrica de avaliação

* **Estatística KS**.
    * Mais detalhes: [Teste de Kolmogorov-Smirnov](https://en.wikipedia.org/wiki/Kolmogorov%E2%80%93Smirnov_test)
* Encontrar o maior KS possível entre as bases de treino e teste.

In [1]:
#Imports
import pandas as pd     #Manipulação dos dados
import numpy as np      #Operações multidimensionais e matemáticas
from scipy import stats #Cálculo do KS
from scipy.stats import ks_2samp #Cálculo do KS
import matplotlib.pyplot as plt #Visualização dos dados
import seaborn as sns #Visualização de dados
##Machine learning
from pycaret.classification import * 
from scipy.stats import ks_2samp
##Eliminar os warnings
import warnings
warnings.filterwarnings("ignore")
##Ver todas as colunas do data frame
pd.set_option('display.max_columns', None)
##Extrair os valores do feature importance
import sklearn as sk

In [2]:
##########################################
##Métrica de Avaliação:
#KS (Kolmogorov-Smirnov)
##########################################
def calcular_ks_2samp(df, alvo, escore):
    bons_escore = df.loc[df[alvo] == 0, escore].rename('bons')
    maus_escore = df.loc[df[alvo] == 1, escore].rename('maus')
    return stats.ks_2samp(bons_escore, maus_escore).statistic

* **OBS**: Se você deseja reproduzir esse código, é recomendado instalar as versões do **Python e do PyCaret indicadas abaixo**.


In [3]:
#Versão do Python: 3.9.13
import sys
print(sys.version)

3.9.13 (main, Oct 13 2022, 21:23:06) [MSC v.1916 64 bit (AMD64)]


In [4]:
#Versão do PyCaret: 3.3.1
!pip show pycaret 

Name: pycaret
Version: 3.3.1
Summary: PyCaret - An open source, low-code machine learning library in Python.
Home-page: https://github.com/pycaret/pycaret
Author: Moez Ali
Author-email: moez.ali@queensu.ca
License: MIT
Location: c:\users\zaian\anaconda3\envs\pycaret_env\lib\site-packages
Requires: category-encoders, cloudpickle, deprecation, imbalanced-learn, importlib-metadata, ipython, ipywidgets, jinja2, joblib, kaleido, lightgbm, markupsafe, matplotlib, nbformat, numba, numpy, pandas, plotly, plotly-resampler, pmdarima, psutil, pyod, requests, schemdraw, scikit-learn, scikit-plot, scipy, sktime, statsmodels, tbats, tqdm, xxhash, yellowbrick
Required-by: 


# 1 - Importação dos dados

**Base de treino**

In [5]:
#Importar os dados
df_treino = pd.read_csv('df_treino.csv', sep = ';', encoding='latin-1')
#Visualização
df_treino.head(4)

,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,taxa_mensal,vlr_renda,vlr_max_renda_presum,pc_compr_renda_familiar,autonomos,taxistas,status_prop,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target
0,obs6218,41,SC,SUL,30,52.93,2021,2 anos,60,28390.79,1.02,3690.0,861.0,18,N,N,EF,677,NaN,NaN,NaN,NaN,S,4,0.0
1,obs623,58,PA,NORTE,30,14.65,2021,0 Km,48,35834.69,1.26,4510.0,1271.0,23,N,N,EF,692,1.0,A2,BOM,C2,NaN,4,0.0
2,obs5136,45,AM,NORTE,30,45.46,2021,0 Km,60,40805.56,1.34,6150.0,1353.0,16,N,N,EF,667,2.0,A2,RUIM,A3,N,4,0.0
3,obs8822,45,SP,SULDESTE,30,28.90,2013,>= 3 anos,36,22291.75,1.30,4510.0,4346.0,18,N,N,EF,698,NaN,NaN,NaN,NaN,S,12,0.0


In [6]:
#Volumetria
df_treino.shape[0] #6.643

6643

In [7]:
#Taxa do ALVO
round(100 * df_treino['target'].value_counts(True),2) #2.98%

0.0    97.03
1.0     2.97
Name: target, dtype: float64

In [8]:
df_treino.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6643 entries, 0 to 6642
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   observacao                6643 non-null   object 
 1   idade                     6643 non-null   int64  
 2   uf                        6643 non-null   object 
 3   regiao                    6643 non-null   object 
 4   carencia                  6643 non-null   int64  
 5   percentual_entrada_veicu  6643 non-null   float64
 6   ano_modelo_veic           6643 non-null   int64  
 7   idade_bem                 6643 non-null   object 
 8   qtde_prestacoes           6643 non-null   int64  
 9   vlr_principal             6643 non-null   float64
 10  taxa_mensal               6643 non-null   float64
 11  vlr_renda                 6643 non-null   float64
 12  vlr_max_renda_presum      6643 non-null   float64
 13  pc_compr_renda_familiar   6643 non-null   int64  
 14  autonomo

**Base de teste**

In [9]:
#Importar os dados
df_teste = pd.read_csv('df_teste.csv', sep = ';', encoding='latin-1')
#Visualização
df_teste.head(4)

,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,taxa_mensal,vlr_renda,vlr_max_renda_presum,pc_compr_renda_familiar,autonomos,taxistas,status_prop,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target
0,obs5043,59,SP,SULDESTE,30,32.76,2022,0 Km,60,29752.78,1.02,8200.0,6683.0,8,N,N,EF,747,3.0,A4,RUIM,B3,S,3,0.0
1,obs3416,39,RO,NORTE,30,42.50,2021,0 Km,49,34698.44,1.42,4920.0,5986.0,22,N,N,EF,669,1.0,A2,RUIM,A3,NaN,4,0.0
2,obs8834,32,PR,SUL,30,67.54,2022,0 Km,24,16245.26,0.80,6150.0,1845.0,12,N,N,EF,746,2.0,A1,BOM,A1,S,3,0.0
3,obs8959,64,PI,NORDESTE,30,56.94,2022,0 Km,36,39508.16,1.30,9840.0,8733.0,14,N,N,EF,683,3.0,A2,RUIM,A1,S,3,0.0


In [10]:
#Volumetria
df_teste.shape[0] #2.848

2848

In [11]:
#Taxa do ALVO
round(100 * df_teste['target'].value_counts(True),2) #2.98%

0.0    97.02
1.0     2.98
Name: target, dtype: float64

In [12]:
df_teste.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2848 entries, 0 to 2847
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   observacao                2848 non-null   object 
 1   idade                     2848 non-null   int64  
 2   uf                        2848 non-null   object 
 3   regiao                    2848 non-null   object 
 4   carencia                  2848 non-null   int64  
 5   percentual_entrada_veicu  2848 non-null   float64
 6   ano_modelo_veic           2848 non-null   int64  
 7   idade_bem                 2848 non-null   object 
 8   qtde_prestacoes           2848 non-null   int64  
 9   vlr_principal             2848 non-null   float64
 10  taxa_mensal               2848 non-null   float64
 11  vlr_renda                 2848 non-null   float64
 12  vlr_max_renda_presum      2848 non-null   float64
 13  pc_compr_renda_familiar   2848 non-null   int64  
 14  autonomo

# 2 - Modelagem

## 2.1 - Setup

* Pre-processamen dos dados:
    * Imputação de valores ausentes,
    * Seleção de variáveis na modelagem,
    * Codificação de variáveis categóricas,
    * Escalonamento dos dados. s.

In [13]:
exp = setup(data = df_treino, test_data = df_teste, target= "target", ignore_features=['idade_veiculo', 'carencia', 'observacao'], normalize= True,
            normalize_method= "minmax", encoding_method= None ,session_id= 1935, experiment_name= "EXP_15_09_2025_V1", index=False)

,Description,Value
0,Session id,1935
1,Target,target
2,Target type,Binary
3,Original data shape,"(9491, 25)"
4,Transformed data shape,"(9491, 41)"
5,Transformed train set shape,"(6643, 41)"
6,Transformed test set shape,"(2848, 41)"
7,Ignore features,3
8,Numeric features,11
9,Categorical features,10


In [14]:
#Volumetria e conjunto de variáveis das bases amostrais
print(f"Base de treino - volumetria e conjunto de variáveis explicativas: {exp.X_train.shape}")
print(f"Base de teste - volumetria e conjunto de variáveis explicativas: {exp.X_test.shape}")
print(f"Y de treinamento target: {exp.y_train.shape}")
print(f"Y de teste: {exp.y_test.shape}")

Base de treino - volumetria e conjunto de variáveis explicativas: (6643, 21)
Base de teste - volumetria e conjunto de variáveis explicativas: (2848, 21)
Y de treinamento target: (6643,)
Y de teste: (2848,)


In [15]:
#Features da base de treino transformada
X = get_config('X_train_transformed')
#Visualização
X.head(3)

,idade,idade_bem_2 anos,idade_bem_0 Km,idade_bem_>= 3 anos,uf,regiao_SUL,regiao_NORTE,regiao_SULDESTE,regiao_NORDESTE,regiao_CENTRO-OESTE,percentual_entrada_veicu,ano_modelo_veic,qtde_prestacoes,vlr_principal,taxa_mensal,vlr_renda,vlr_max_renda_presum,pc_compr_renda_familiar,autonomos,taxistas,status_prop_EF,score_sis,var_1,var_2_A2,var_2_A4,var_2_A1,var_2_A0,var_3_BOM,var_3_RUIM,var_3_PESS,var_4_A1,var_4_C2,var_4_A3,var_4_A2,var_4_B1,var_4_C3,var_4_B3,var_4_B2,var_4_C1,var_5
0,0.295775,1.0,0.0,0.0,0.728520,1.0,0.0,0.0,0.0,0.0,0.558805,0.9,1.00,0.165004,0.398438,0.000109,0.040000,0.183673,0.0,0.0,0.0,0.421965,0.727027,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.535211,0.0,1.0,0.0,0.052925,0.0,1.0,0.0,0.0,0.0,0.154666,0.9,0.75,0.217442,0.492188,0.000133,0.059048,0.234694,0.0,0.0,0.0,0.508671,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.352113,0.0,1.0,0.0,0.381280,0.0,1.0,0.0,0.0,0.0,0.479941,0.9,1.00,0.252459,0.523438,0.000181,0.062857,0.163265,0.0,0.0,0.0,0.364162,0.500000,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2.2 - Compare models

* Comparar diferentes modelos de machine learning. Esta função treina e avalia todos os modelos disponíveis na biblioteca usando validação cruzada. Ela retorna uma tabela de desempenho, permitindo que você identifique o melhor modelo para o seu conjunto de dados.

In [16]:
compare_models() 

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
et,Extra Trees Classifier,0.9708,0.0000,0.0463,0.6000,0.0853,0.0815,0.1586,0.3580
rf,Random Forest Classifier,0.9706,0.0000,0.0103,0.2000,0.0195,0.0190,0.0447,0.4700
lr,Logistic Regression,0.9703,0.8164,0.0000,0.0000,0.0000,0.0000,0.0000,1.2650
svm,SVM - Linear Kernel,0.9703,0.7695,0.0000,0.0000,0.0000,0.0000,0.0000,0.1720
ridge,Ridge Classifier,0.9703,0.7848,0.0000,0.0000,0.0000,0.0000,0.0000,0.1830
lightgbm,Light Gradient Boosting Machine,0.9703,0.0000,0.0671,0.4267,0.1113,0.1057,0.1544,0.4860
catboost,CatBoost Classifier,0.9703,0.0000,0.0511,0.4000,0.0899,0.0852,0.1352,4.1500
dummy,Dummy Classifier,0.9703,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1800
lda,Linear Discriminant Analysis,0.9700,0.7845,0.0000,0.0000,0.0000,-0.0006,-0.0014,0.1980
gbc,Gradient Boosting Classifier,0.9697,0.8177,0.0666,0.3467,0.1095,0.1030,0.1409,0.9910


ExtraTreesClassifier(bootstrap=False, ccp_alpha=0.0, class_weight=None,
                     criterion='gini', max_depth=None, max_features='sqrt',
                     max_leaf_nodes=None, max_samples=None,
                     min_impurity_decrease=0.0, min_samples_leaf=1,
                     min_samples_split=2, min_weight_fraction_leaf=0.0,
                     monotonic_cst=None, n_estimators=100, n_jobs=-1,
                     oob_score=False, random_state=1935, verbose=0,
                     warm_start=False)

## 2.3 - Create model

* Treina um modelo de machine learning específico.
* **Comparar os KS das bases de treino e teste**.
  * Quanto mais próximos estes KS, menor o overfitting associado a modelagem.

### A - Gradient Boosting

In [17]:
model_1 = create_model('gbc') 

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9699,0.8022,0.0000,0.0000,0.0000,0.0000,0.0000
1,0.9684,0.8540,0.1000,0.4000,0.1600,0.1498,0.1885
2,0.9684,0.8357,0.1000,0.4000,0.1600,0.1498,0.1885
3,0.9714,0.8192,0.0526,0.5000,0.0952,0.0903,0.1554
4,0.9714,0.8070,0.1579,0.5000,0.2400,0.2294,0.2700
5,0.9729,0.8472,0.1053,0.6667,0.1818,0.1754,0.2578
6,0.9684,0.8110,0.0000,0.0000,0.0000,-0.0029,-0.0068
7,0.9699,0.8477,0.0500,0.5000,0.0909,0.0859,0.1511
8,0.9699,0.7849,0.1000,0.5000,0.1667,0.1582,0.2140


In [18]:
#Avaliar o modelo na base de teste
##Criar um data frame
df_teste_1 = predict_model(model_1, data = df_teste)
#Visualização
df_teste_1.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Gradient Boosting Classifier,0.9680,0.8729,0.0471,0.2857,0.0808,0.0730,0.1057


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs5043,59,SP,SULDESTE,30,32.759998,2022,0 Km,60,29752.779297,...,747,3.0,A4,RUIM,B3,S,3,0.0,0,0.9965
1,obs3416,39,RO,NORTE,30,42.500000,2021,0 Km,49,34698.441406,...,669,1.0,A2,RUIM,A3,NaN,4,0.0,0,0.9833
2,obs8834,32,PR,SUL,30,67.540001,2022,0 Km,24,16245.259766,...,746,2.0,A1,BOM,A1,S,3,0.0,0,0.9971


In [19]:
#Aplicar o ks de teste
calcular_ks_2samp(df_teste_1, 'target', 'prediction_score') #0.5882778735815716

0.5882778735815716

In [20]:
#Avaliar o modelo na base de treino
##Criar um data frame
df_treino_1 = predict_model(model_1, data = df_treino)
#Visualização
df_treino_1.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Gradient Boosting Classifier,0.9768,0.9423,0.2183,1.0000,0.3583,0.3514,0.4617


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs6218,41,SC,SUL,30,52.930000,2021,2 anos,60,28390.789062,...,677,NaN,NaN,NaN,NaN,S,4,0.0,0,0.9919
1,obs623,58,PA,NORTE,30,14.650000,2021,0 Km,48,35834.691406,...,692,1.0,A2,BOM,C2,NaN,4,0.0,0,0.9640
2,obs5136,45,AM,NORTE,30,45.459999,2021,0 Km,60,40805.558594,...,667,2.0,A2,RUIM,A3,N,4,0.0,0,0.9807


In [21]:
#Aplicar o ks de treino
calcular_ks_2samp(df_treino_1, 'target', 'prediction_score') #0.7246685072866186

0.7246685072866186

### B - Random Forest

In [22]:
model_2 = create_model('rf') 

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9699,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,0.9699,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,0.9699,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,0.9714,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,0.9714,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5,0.9729,0.0000,0.0526,1.0000,0.1000,0.0974,0.2263
6,0.9699,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
7,0.9714,0.0000,0.0500,1.0000,0.0952,0.0926,0.2204
8,0.9699,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [23]:
#Avaliar o modelo na base de teste
##Criar um data frame
df_teste_2 = predict_model(model_2, data = df_teste)
#Visualização
df_teste_2.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.9716,0.8622,0.0471,1.0000,0.0899,0.0874,0.2138


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs5043,59,SP,SULDESTE,30,32.759998,2022,0 Km,60,29752.779297,...,747,3.0,A4,RUIM,B3,S,3,0.0,0,1.00
1,obs3416,39,RO,NORTE,30,42.500000,2021,0 Km,49,34698.441406,...,669,1.0,A2,RUIM,A3,NaN,4,0.0,0,0.98
2,obs8834,32,PR,SUL,30,67.540001,2022,0 Km,24,16245.259766,...,746,2.0,A1,BOM,A1,S,3,0.0,0,1.00


In [24]:
#Aplicar o ks de teste
calcular_ks_2samp(df_teste_2, 'target', 'prediction_score') #0.5799493304379297

0.5799493304379297

In [25]:
#Avaliar o modelo na base de treino
##Criar um data frame
df_treino_2 = predict_model(model_2, data = df_treino)
#Visualização
df_treino_2.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs6218,41,SC,SUL,30,52.930000,2021,2 anos,60,28390.789062,...,677,NaN,NaN,NaN,NaN,S,4,0.0,0,1.00
1,obs623,58,PA,NORTE,30,14.650000,2021,0 Km,48,35834.691406,...,692,1.0,A2,BOM,C2,NaN,4,0.0,0,0.97
2,obs5136,45,AM,NORTE,30,45.459999,2021,0 Km,60,40805.558594,...,667,2.0,A2,RUIM,A3,N,4,0.0,0,0.99


In [26]:
#Aplicar o ks de treino
calcular_ks_2samp(df_treino_2, 'target', 'prediction_score') #0.995035681042507

0.995035681042507

### C - Regressão Logística

In [27]:
model_3 = create_model('lr') 

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9699,0.7745,0.0000,0.0000,0.0000,0.0000,0.0000
1,0.9699,0.8367,0.0000,0.0000,0.0000,0.0000,0.0000
2,0.9699,0.8264,0.0000,0.0000,0.0000,0.0000,0.0000
3,0.9714,0.8228,0.0000,0.0000,0.0000,0.0000,0.0000
4,0.9714,0.8113,0.0000,0.0000,0.0000,0.0000,0.0000
5,0.9714,0.8684,0.0000,0.0000,0.0000,0.0000,0.0000
6,0.9699,0.8286,0.0000,0.0000,0.0000,0.0000,0.0000
7,0.9699,0.7807,0.0000,0.0000,0.0000,0.0000,0.0000
8,0.9699,0.8031,0.0000,0.0000,0.0000,0.0000,0.0000


In [28]:
#Avaliar o modelo na base de teste
##Criar um data frame
df_teste_3 = predict_model(model_3, data = df_teste)
#Visualização
df_teste_3.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.9702,0.8523,0.0000,0.0000,0.0000,0.0000,0.0000


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs5043,59,SP,SULDESTE,30,32.759998,2022,0 Km,60,29752.779297,...,747,3.0,A4,RUIM,B3,S,3,0.0,0,0.9967
1,obs3416,39,RO,NORTE,30,42.500000,2021,0 Km,49,34698.441406,...,669,1.0,A2,RUIM,A3,NaN,4,0.0,0,0.9730
2,obs8834,32,PR,SUL,30,67.540001,2022,0 Km,24,16245.259766,...,746,2.0,A1,BOM,A1,S,3,0.0,0,0.9983


In [29]:
#Aplicar o ks de teste
calcular_ks_2samp(df_teste_3, 'target', 'prediction_score') #0.5481041493687594

0.5481041493687594

In [30]:
#Avaliar o modelo na base de treino
##Criar um data frame
df_treino_3 = predict_model(model_3, data = df_treino)
#Visualização
df_treino_3.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.9703,0.8510,0.0000,0.0000,0.0000,0.0000,0.0000


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs6218,41,SC,SUL,30,52.930000,2021,2 anos,60,28390.789062,...,677,NaN,NaN,NaN,NaN,S,4,0.0,0,0.9689
1,obs623,58,PA,NORTE,30,14.650000,2021,0 Km,48,35834.691406,...,692,1.0,A2,BOM,C2,NaN,4,0.0,0,0.9785
2,obs5136,45,AM,NORTE,30,45.459999,2021,0 Km,60,40805.558594,...,667,2.0,A2,RUIM,A3,N,4,0.0,0,0.9597


In [31]:
#Aplicar o ks de treino
calcular_ks_2samp(df_treino_3, 'target', 'prediction_score') #0.551281950322161

0.551281950322161

### D - Regressão Logística ponderada

* Ajusta automaticamente os pesos inversamente proporcionais à frequência das classes.

* Classes minoritárias recebem pesos maiores.

* Classes majoritárias recebem pesos menores.

* Melhor para dados desbalanceados.

In [32]:
model_4 = create_model('lr', class_weight='balanced') 

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7203,0.7614,0.7000,0.0722,0.1308,0.0807,0.1582
1,0.7368,0.8461,0.8000,0.0856,0.1546,0.1060,0.2032
2,0.7444,0.8332,0.7500,0.0833,0.1500,0.1014,0.1900
3,0.7455,0.8213,0.5789,0.0640,0.1152,0.0671,0.1253
4,0.7319,0.7961,0.7368,0.0749,0.1359,0.0886,0.1737
5,0.7093,0.8561,0.8421,0.0777,0.1422,0.0948,0.1973
6,0.7349,0.8316,0.8000,0.0851,0.1538,0.1051,0.2022
7,0.7395,0.7701,0.6500,0.0726,0.1307,0.0808,0.1511
8,0.7364,0.7866,0.7000,0.0765,0.1379,0.0884,0.1674


In [33]:
#Avaliar o modelo na base de teste
##Criar um data frame
df_teste_4 = predict_model(model_4, data = df_teste)
#Visualização
df_teste_4.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.7381,0.8469,0.7765,0.0832,0.1503,0.1019,0.1949


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs5043,59,SP,SULDESTE,30,32.759998,2022,0 Km,60,29752.779297,...,747,3.0,A4,RUIM,B3,S,3,0.0,0,0.9370
1,obs3416,39,RO,NORTE,30,42.500000,2021,0 Km,49,34698.441406,...,669,1.0,A2,RUIM,A3,NaN,4,0.0,0,0.5958
2,obs8834,32,PR,SUL,30,67.540001,2022,0 Km,24,16245.259766,...,746,2.0,A1,BOM,A1,S,3,0.0,0,0.9641


In [34]:
#Aplicar o ks de teste
calcular_ks_2samp(df_teste_4, 'target', 'prediction_score') #0.22493027612782354

0.22493027612782354

In [35]:
#Avaliar o modelo na base de treino
##Criar um data frame
df_treino_4 = predict_model(model_4, data = df_treino)
#Visualização
df_treino_4.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.7364,0.8547,0.8122,0.0854,0.1545,0.1066,0.2059


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs6218,41,SC,SUL,30,52.930000,2021,2 anos,60,28390.789062,...,677,NaN,NaN,NaN,NaN,S,4,0.0,0,0.6267
1,obs623,58,PA,NORTE,30,14.650000,2021,0 Km,48,35834.691406,...,692,1.0,A2,BOM,C2,NaN,4,0.0,0,0.6492
2,obs5136,45,AM,NORTE,30,45.459999,2021,0 Km,60,40805.558594,...,667,2.0,A2,RUIM,A3,N,4,0.0,1,0.5435


In [36]:
#Aplicar o ks de treino
calcular_ks_2samp(df_treino_4, 'target', 'prediction_score') #0.23350726299393162

0.23350726299393162

### E - Extra Tree

In [37]:
model_5 = create_model('et') 

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9699,0.0000,0.0500,0.5000,0.0909,0.0859,0.1511
1,0.9699,0.0000,0.0500,0.5000,0.0909,0.0859,0.1511
2,0.9714,0.0000,0.0500,1.0000,0.0952,0.0926,0.2204
3,0.9729,0.0000,0.0526,1.0000,0.1000,0.0974,0.2263
4,0.9744,0.0000,0.1053,1.0000,0.1905,0.1860,0.3202
5,0.9744,0.0000,0.1053,1.0000,0.1905,0.1860,0.3202
6,0.9714,0.0000,0.0500,1.0000,0.0952,0.0926,0.2204
7,0.9684,0.0000,0.0000,0.0000,0.0000,-0.0029,-0.0068
8,0.9669,0.0000,0.0000,0.0000,0.0000,-0.0055,-0.0097


In [38]:
#Avaliar o modelo na base de teste
##Criar um data frame
df_teste_5 = predict_model(model_5, data = df_teste)
#Visualização
df_teste_5.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,0.9712,0.8540,0.0471,0.8000,0.0889,0.0859,0.1898


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs5043,59,SP,SULDESTE,30,32.759998,2022,0 Km,60,29752.779297,...,747,3.0,A4,RUIM,B3,S,3,0.0,0,1.00
1,obs3416,39,RO,NORTE,30,42.500000,2021,0 Km,49,34698.441406,...,669,1.0,A2,RUIM,A3,NaN,4,0.0,0,0.97
2,obs8834,32,PR,SUL,30,67.540001,2022,0 Km,24,16245.259766,...,746,2.0,A1,BOM,A1,S,3,0.0,0,1.00


In [39]:
#Aplicar o ks de teste
calcular_ks_2samp(df_teste_5, 'target', 'prediction_score') #0.5236763109152456

0.5236763109152456

In [40]:
#Avaliar o modelo na base de treino
##Criar um data frame
df_treino_5 = predict_model(model_5, data = df_treino)
#Visualização
df_treino_5.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs6218,41,SC,SUL,30,52.930000,2021,2 anos,60,28390.789062,...,677,NaN,NaN,NaN,NaN,S,4,0.0,0,1.0
1,obs623,58,PA,NORTE,30,14.650000,2021,0 Km,48,35834.691406,...,692,1.0,A2,BOM,C2,NaN,4,0.0,0,1.0
2,obs5136,45,AM,NORTE,30,45.459999,2021,0 Km,60,40805.558594,...,667,2.0,A2,RUIM,A3,N,4,0.0,0,1.0


In [41]:
#Aplicar o ks de teste
calcular_ks_2samp(df_treino_5, 'target', 'prediction_score') #0

0.0

## 2.4 - Tune Model

* Regressão Logística

In [42]:
model_3_1 = tune_model(model_3)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9699,0.7740,0.0000,0.0000,0.0000,0.0000,0.0000
1,0.9699,0.8389,0.0000,0.0000,0.0000,0.0000,0.0000
2,0.9699,0.8266,0.0000,0.0000,0.0000,0.0000,0.0000
3,0.9714,0.8230,0.0000,0.0000,0.0000,0.0000,0.0000
4,0.9714,0.8131,0.0000,0.0000,0.0000,0.0000,0.0000
5,0.9714,0.8671,0.0000,0.0000,0.0000,0.0000,0.0000
6,0.9699,0.8292,0.0000,0.0000,0.0000,0.0000,0.0000
7,0.9699,0.7814,0.0000,0.0000,0.0000,0.0000,0.0000
8,0.9699,0.8009,0.0000,0.0000,0.0000,0.0000,0.0000


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [43]:
#Avaliar o modelo na base de teste
##Criar um data frame
df_teste_3_1 = predict_model(model_3_1, data = df_teste)
#Visualização
df_teste_3_1.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.9702,0.8523,0.0000,0.0000,0.0000,0.0000,0.0000


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs5043,59,SP,SULDESTE,30,32.759998,2022,0 Km,60,29752.779297,...,747,3.0,A4,RUIM,B3,S,3,0.0,0,0.9967
1,obs3416,39,RO,NORTE,30,42.500000,2021,0 Km,49,34698.441406,...,669,1.0,A2,RUIM,A3,NaN,4,0.0,0,0.9730
2,obs8834,32,PR,SUL,30,67.540001,2022,0 Km,24,16245.259766,...,746,2.0,A1,BOM,A1,S,3,0.0,0,0.9983


In [44]:
#Aplicar o ks de teste
calcular_ks_2samp(df_teste_3_1, 'target', 'prediction_score') #0.5481041493687594

0.5481041493687594

In [45]:
#Avaliar o modelo na base de treino
##Criar um data frame
df_treino_3_1 = predict_model(model_3_1, data = df_treino)
#Visualização
df_treino_3_1.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.9703,0.8510,0.0000,0.0000,0.0000,0.0000,0.0000


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,score_sis,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score
0,obs6218,41,SC,SUL,30,52.930000,2021,2 anos,60,28390.789062,...,677,NaN,NaN,NaN,NaN,S,4,0.0,0,0.9689
1,obs623,58,PA,NORTE,30,14.650000,2021,0 Km,48,35834.691406,...,692,1.0,A2,BOM,C2,NaN,4,0.0,0,0.9785
2,obs5136,45,AM,NORTE,30,45.459999,2021,0 Km,60,40805.558594,...,667,2.0,A2,RUIM,A3,N,4,0.0,0,0.9597


In [46]:
#Aplicar o ks de treino
calcular_ks_2samp(df_treino_3_1, 'target', 'prediction_score') #0.551281950322161

0.551281950322161

## 2.5 - Avaliar os resultados na base de teste

* Mais detalhes da performance do modelo
* Modelo escolhido - Regressão Logística (**model_3**)

In [47]:
dashboard(model_3, display_format='dash')

Note: model_output='probability' is currently not supported for linear classifiers models with shap. So defaulting to model_output='logodds' If you really need probability outputs use shap='kernel' instead.
Note: shap values for shap='linear' get calculated against X_background, but paramater X_background=None, so using X instead...
Generating self.shap_explainer = shap.LinearExplainer(model, X)...
Building ExplainerDashboard..
Detected notebook environment, consider setting mode='external', mode='inline' or mode='jupyterlab' to keep the notebook interactive while the dashboard is running...
For this type of model and model_output interactions don't work, so setting shap_interaction=False...
The explainer object has no decision_trees property. so setting decision_trees=False...
Generating layout...
Calculating shap values...
Calculating prediction probabilities...
Calculating metrics...
Calculating confusion matrices...
Calculating classification_dfs...
Calculating roc auc curves...
Ca

192.168.1.72 - - [21/Sep/2025 11:01:38] code 400, message Bad request version ('zz\x13\x01\x13\x02\x13\x03À+À/À,À0Ì©Ì¨À\x13À\x14\x00\x9c\x00\x9d\x00/\x005\x01\x00\x06s')
192.168.1.72 - - [21/Sep/2025 11:01:38] code 400, message Bad request version ('\x00\x02\x01\x00\x00')


ConnectionError: HTTPConnectionPool(host='0.0.0.0', port=8050): Max retries exceeded with url: /_alive_be814946-285e-4400-8bab-bc086651fbc3 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001C9F01428E0>: Failed to establish a new connection: [WinError 10049] O endereço solicitado não é válido no contexto'))

### Avaliação de desempenho

In [48]:
#Top 10 variáveis
plot_model(model_3, plot = 'feature', save=True)

'Feature Importance.png'

In [50]:
#Ordem de importância de todas as variáveis
plot_model(model_3, plot = 'feature_all', save=True)

'Feature Importance (All).png'

In [51]:
#Panorama geral dos resultados
evaluate_model(model_3)

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [56]:
# Criar DataFrame detalhado
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model_3.coef_[0],        # Coeficiente original
    'abs_coefficient': np.abs(model_3.coef_[0]),  # Valor absoluto
    'importance_rank': np.argsort(np.abs(model_3.coef_[0]))[::-1] + 1  # Ranking
})

# Ordenar por importância
feature_importance = feature_importance.sort_values('abs_coefficient', ascending=False)
print(feature_importance)

                     feature  coefficient  abs_coefficient  importance_rank
21                 score_sis    -3.295950         3.295950                4
14               taxa_mensal     2.479543         2.479543               40
10  percentual_entrada_veicu    -1.856060         1.856060               18
4                         uf     1.700951         1.700951               17
16      vlr_max_renda_presum    -1.667595         1.667595                3
26                  var_2_A0    -1.348294         1.348294               19
13             vlr_principal     0.802525         0.802525               39
0                      idade    -0.789656         0.789656               22
22                     var_1    -0.753674         0.753674               33
15                 vlr_renda     0.753607         0.753607               35
17   pc_compr_renda_familiar     0.659043         0.659043               31
35                  var_4_C3     0.593210         0.593210               30
27          

## 2.6 - Concluir o experimento

In [57]:
#Finalizar o modelo
##O modelo vai ser treinado com todo o data set (base completa), considerando a regressão lasso.
final_model = finalize_model(model_3)
final_model

Pipeline(memory=Memory(location=None),
         steps=[('numerical_imputer',
                 TransformerWrapper(exclude=None,
                                    include=['idade',
                                             'percentual_entrada_veicu',
                                             'ano_modelo_veic',
                                             'qtde_prestacoes', 'vlr_principal',
                                             'taxa_mensal', 'vlr_renda',
                                             'vlr_max_renda_presum',
                                             'pc_compr_renda_familiar',
                                             'score_sis', 'var_1'],
                                    transformer=SimpleImputer(add_indicator=False,
                                                              copy=True,
                                                              fill...
                 TransformerWrapper(exclude=None, include=None,
                                    transformer=MinMaxScaler(clip=False,
                                                             copy=True,
                                                             feature_range=(0,
                                                                            1)))),
                ('actual_estimator',
                 LogisticRegression(C=1.0, class_weight=None, dual=False,
                                    fit_intercept=True, intercept_scaling=1,
                                    l1_ratio=None, max_iter=1000,
                                    multi_class='auto', n_jobs=None,
                                    penalty='l2', random_state=1935,
                                    solver='lbfgs', tol=0.0001, verbose=0,
                                    warm_start=False))],
         verbose=False)

In [58]:
previsoes_df = predict_model(final_model, data=df_teste, raw_score=True)
#Visualização
previsoes_df.head(3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.9698,0.8724,0.0000,0.0000,0.0000,-0.0007,-0.0033


,observacao,idade,uf,regiao,carencia,percentual_entrada_veicu,ano_modelo_veic,idade_bem,qtde_prestacoes,vlr_principal,...,var_1,var_2,var_3,var_4,var_5,idade_veiculo,target,prediction_label,prediction_score_0,prediction_score_1
0,obs5043,59,SP,SULDESTE,30,32.759998,2022,0 Km,60,29752.779297,...,3.0,A4,RUIM,B3,S,3,0.0,0,0.9969,0.0031
1,obs3416,39,RO,NORTE,30,42.500000,2021,0 Km,49,34698.441406,...,1.0,A2,RUIM,A3,NaN,4,0.0,0,0.9650,0.0350
2,obs8834,32,PR,SUL,30,67.540001,2022,0 Km,24,16245.259766,...,2.0,A1,BOM,A1,S,3,0.0,0,0.9987,0.0013


In [59]:
#Remomear a variável prediction_label
previsoes_df.rename(columns = {'prediction_score_0':'escore'}, inplace = True)

In [60]:
tabela_final = previsoes_df[['observacao', 'regiao', 'target','escore']]
#Visualização
tabela_final.head(3)

,observacao,regiao,target,escore
0,obs5043,SULDESTE,0.0,0.9969
1,obs3416,NORTE,0.0,0.9650
2,obs8834,SUL,0.0,0.9987


In [61]:
#Salvar o data frame
tabela_final.to_csv('df_estimativas_v1.csv', header=True, index=False, sep=';')

In [62]:
#Salvar o modelo ("CPF" do modelo)
save_model(final_model, 'regressao_logistica_1')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['idade',
                                              'percentual_entrada_veicu',
                                              'ano_modelo_veic',
                                              'qtde_prestacoes', 'vlr_principal',
                                              'taxa_mensal', 'vlr_renda',
                                              'vlr_max_renda_presum',
                                              'pc_compr_renda_familiar',
                                              'score_sis', 'var_1'],
                                     transformer=SimpleImputer(add_indicator=False,
                                                               copy=True,
                                                               fill...
                  TransformerWrapper(exclude=None, include=None,
              

In [63]:
save_experiment('my_saved_experiment')